# Sorting with multi-head attention

A transformer that learns to sort a sequence of digits. The model uses
learned embeddings, sinusoidal positional encoding, multi-head causal
self-attention, and layer normalization (Pre-LN architecture).

**Task:** Given `[3,1,4,0,2]`, predict `[0,1,2,3,4]` (with separator/EOS tokens).

**Reference:** Vaswani et al., "Attention Is All You Need" (2017)

**CLI equivalent:** `make example-transformer` (1000 epochs, batch=16)


## Architecture

The transformer maps a sequence of token indices to logits over the vocabulary.
All dimensions are checked at compile time.


In [ ]:
:browse Nn.Transformer

In [ ]:
:t TransformerBlock

The transformer is decomposed, not a monolith: `TransformerBlock dModel numHeads
headDim` is one pre-norm block, and the compiled example stacks `NumBlocks` of
them (plus a final `LayerNorm`) into a `Seq`, with the embedding, positional
encoding, and output head as ordinary record fields around it:

```idris
record TfmModel (0 ex : Executor) (0 dt : DType) (0 g : GradMode) where
  constructor MkTfmModel
  embed  : Embedding VocabSize DModel ex dt g
  1 body : Seq DModel DModel ex dt g
  headW : TMat VocabSize DModel ex dt g
  pe    : Tensor [SeqLen, DModel] ex dt g
```

The forward maps one `[SeqLen]` token sequence to `[SeqLen, VocabSize]`
per-position logits, threading the linear `body` through `forwardSeq`.

### Sequence format

For sorting 5 digits from vocab {0..5}:
```
Input:  [3, 1, 4, 0, 2, SEP, 0, 1, 2, 3, 4]  (teacher-forced)
Target: predict next token at each position
```
SEP=6, EOS=7, so vocab size is 8. Loss is only computed on the sorting
portion (after SEP).


## Training

The compiled example uses a small config (dModel=16, 4 heads of dim 4, 2
blocks) and trains for 1000 epochs with Adam under `NormClip 1.0`. Fresh
batches of random sorting problems are generated each epoch, and the batch
loss is handed to `fitSupervised`:

```idris
(MkBang (epochsDone, finalLoss) # trained) <-
  fitSupervised {ex=ExampleExecutor} opt batchLossL
                (generate (sortingBatch BatchSize)) trainCfg model
```

The attention primitive under the blocks:


In [ ]:
:t attention

## Key components

### Multi-head attention
Each head computes `softmax(QK^T / sqrt(d_k)) V` independently, then the
per-head output projections are summed (not concatenated), keeping the
output dimension at `dModel`.

### Causal mask
The attention mask prevents positions from attending to future tokens,
enabling autoregressive generation.

### Pre-LN vs Post-LN
Layer normalization is applied *before* attention and FFN (Pre-LN),
which is more stable than the original Post-LN architecture.


## PyTorch comparison

```python
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, 4*d_model),
                                nn.GELU(), nn.Linear(4*d_model, d_model))
```

In idris-ml, `transformerBlock` builds the same block in the `Init` monad, and
the type system checks that `numHeads * headDim` agrees with `dModel` and that
the `[SeqLen, VocabSize]` output shape is what the loss consumes.

See `pytorch/torch_ref/scripts/transformer.py` for the full reference.


Next: [GPT](gpt.ipynb) — character-level language model using the same transformer.
